In [1]:
import os
from utils import print_completion_stats, load_experiment_results

os.environ['OPENESTIMATE_ROOT'] = '/Users/marzoev/openestimate'
experiment_name = 'model_family_comparison'
datasets = ['pitchbook']
results_sets = [load_experiment_results(dataset, experiment_name) for dataset in datasets] 
for results, variables in results_sets: 
        print_completion_stats(results)

results = results_sets[0][0]
variables = results_sets[0][1]

Number of trials found:  1
Completion Statistics

Completion-rate by model (%):

Approach                                   Completion %
-----------------------------------------------------------------
gpt-4o_base_direct_temp0.5                  100.0
meta-llama-3-70b_base_direct_temp0.5        100.0
meta-llama-3-8b_base_direct_temp0.5         100.0
o3-mini_base_direct_tempmedium              100.0
o4-mini_base_direct_tempmedium              100.0


# Load baselines 

In [2]:
import json 

baselines_file_path = os.path.expanduser("~/openestimate/data/baselines/{}_baselines.json".format('pitchbook')) 
stat_baselines = json.load(open(baselines_file_path, 'r'))


In [3]:
import pandas as pd 
from scipy.stats import beta as beta_dist, norm as normal_dist


stat_baselines_metrics_by_var = {}
for var, info in stat_baselines.items():
    stat_baselines_metrics_by_var[var] = {}
    for n, baseline_infos in info.items():
        var_name = variables[var]['variable']

        # Store each resampling's metrics separately
        resampling_results = []

        # Determine if this is a lognormal baseline
        is_lognormal = 'lognorm' in str(n)
        # Extract the numeric sample size (e.g., "5" from "5_lognorm" or "5" from "5")
        n_numeric = str(n).replace('_lognorm', '')

        # Process each resampling
        trial = 0
        for baseline_info in baseline_infos:
            var_type = 'beta' if 'alpha' in baseline_info else 'gaussian'
            if var_type == 'beta':
                mean = beta_dist.mean(baseline_info['alpha'], baseline_info['beta'])
                std = beta_dist.std(baseline_info['alpha'], baseline_info['beta'])
            elif var_type == 'gaussian':
                # For lognormal, don't set mean/std here - they'll be computed later
                if is_lognormal:
                    mean = None
                    std = None
                else:
                    mean = baseline_info['mu']
                    if 'sigma' in baseline_info:
                        std = baseline_info['sigma']
                    else: 
                        std = baseline_info['std']

            var_difficulty = 'easy' if 'easy' in var else 'medium' if 'medium' in var else 'hard' if 'hard' in var else 'base'
            ground_truth = variables[var]['mean']

            # Determine ground truth distribution type based on the baseline type
            if is_lognormal:
                ground_truth_dist = 'lognormal'
            elif var_type == 'gaussian':
                ground_truth_dist = 'gaussian'
            else:
                ground_truth_dist = 'beta'

            # Store each resampling's results separately
            resampling_results.append({
                'variable_name': var_name,
                'variable': var,
                'ground_truth': ground_truth,
                'ground_truth_distribution_type': ground_truth_dist,
                'fitted_distribution_type': ground_truth_dist,
                'mean': mean,
                'std': std,
                'mu': baseline_info.get('mu') if var_type == 'gaussian' else None,
                'sigma': baseline_info.get('sigma', baseline_info.get('std')) if var_type == 'gaussian' else None,
                'a': baseline_info['alpha'] if 'alpha' in baseline_info else None,
                'b': baseline_info['beta'] if 'beta' in baseline_info else None,
                'trial': trial,
                'difficulty': var_difficulty,
                'sample_size': n_numeric
            })
            trial += 1
        # Store all resampling results for this sample size
        stat_baselines_metrics_by_var[var][n] = resampling_results

flattened_metrics = []
for var, baselines in stat_baselines_metrics_by_var.items():
    for n, resamplings in baselines.items():
        # Extract numeric sample size (works for both "5" and "5_lognorm")
        n_numeric = str(n).replace('_lognorm', '')

        for resampling in resamplings:
            resampling_copy = resampling.copy()
            resampling_copy['approach'] = f'statistical_baseline_n{n_numeric}'
            flattened_metrics.append(resampling_copy)

results_df = pd.DataFrame(flattened_metrics)

results = pd.concat([results, results_df], ignore_index=True)

In [4]:
results_df['ground_truth_distribution_type'].value_counts()

ground_truth_distribution_type
gaussian     4040
lognormal    4040
beta         2121
Name: count, dtype: int64

In [5]:
import pandas as pd 

mask = results['fitted_distribution_type'] == 'gaussian'
results.loc[mask, 'mu'] = results.loc[mask, 'mean']
results.loc[mask, 'sigma'] = results.loc[mask, 'std']


results[(results['fitted_distribution_type'] == 'gaussian')].head()
results = results.drop('mean', axis=1)
results = results.drop('std', axis=1)

# Update ground truth column to be chosen based on fitted distribution for LLMs 

In [6]:
mask = results['fitted_distribution_type'] == 'lognormal'

results.loc[mask, 'ground_truth'] = results.loc[mask].apply(
    lambda row: variables[row['variable']].get('mean_lognormal', variables[row['variable']]['mean']), 
    axis=1
)

# Add mean, median, mode of each distribution 

In [7]:
import numpy as np
from scipy import stats


def compute_normal_mean_median_mode(mu, sigma):
    return {'mean': mu, 'median': mu, 'mode': mu}


def compute_beta_mean_median_mode(a, b): 
    # Mean of Beta(a, b) = a / (a + b)
    mean = a / (a + b)
    
    # Median - no closed form, use scipy
    median = stats.beta.median(a, b)
    
    if a > 1 and b > 1:
        mode = (a - 1) / (a + b - 2)
    elif a == 1 and b == 1:
        mode = np.nan  # uniform distribution
    elif a <= 1 and b > 1:
        mode = 0.0
    elif a > 1 and b <= 1:
        mode = 1.0
    else:  # a < 1 and b < 1 (bimodal)
        mode = np.nan  # or could return [0.0, 1.0

    return {
        'mean': mean,
        'median': median,
        'mode': mode  
    }


def compute_lognormal_mean_median_mode(mu, sigma):
    """
    Compute mean, median, and mode of a Lognormal distribution.
    
    Parameters:
    -----------
    mu : float
        Mean of the underlying normal distribution
    sigma : float
        Standard deviation of the underlying normal distribution, must be > 0
    
    Returns:
    --------
    dict with keys 'mean', 'median', 'mode'
    """
    import numpy as np
    
    # Mean of Lognormal(mu, sigma) = exp(mu + sigma^2/2)
    mean = np.exp(mu + sigma**2 / 2)
    
    # Median of Lognormal(mu, sigma) = exp(mu)
    median = np.exp(mu)
    
    # Mode of Lognormal(mu, sigma) = exp(mu - sigma^2)
    mode = np.exp(mu - sigma**2)
    
    return {
        'mean': mean,
        'median': median,
        'mode': mode
    }

In [8]:
beta_vars = results['fitted_distribution_type'] == 'beta'
lognorm_vars = results['fitted_distribution_type'] == 'lognormal'
norm_vars = results['fitted_distribution_type'] == 'gaussian'

results['mean'] = np.nan
results['median'] = np.nan 
results['mode'] = np.nan 

results.loc[beta_vars, ['mean', 'median', 'mode']] = results.loc[beta_vars].apply(
    lambda row: pd.Series(compute_beta_mean_median_mode(row['a'], row['b'])), axis=1)

results.loc[lognorm_vars, ['mean', 'median', 'mode']] = results.loc[lognorm_vars].apply(
    lambda row: pd.Series(compute_lognormal_mean_median_mode(row['mu'], row['sigma'])), axis=1)

results.loc[norm_vars, ['mean', 'median', 'mode']] = results.loc[norm_vars].apply(
    lambda row: pd.Series(compute_normal_mean_median_mode(row['mu'], row['sigma'])), axis=1)

/var/folders/yt/njjd4lrj39q3x45zw301h6h80000gn/T/ipykernel_23257/2410795302.py:52: RuntimeWarning: overflow encountered in exp
  mean = np.exp(mu + sigma**2 / 2)
/var/folders/yt/njjd4lrj39q3x45zw301h6h80000gn/T/ipykernel_23257/2410795302.py:55: RuntimeWarning: overflow encountered in exp
  median = np.exp(mu)


# Compute AE using all central tendencies for each variable 

In [9]:
variables

{'IsUSBased': {'mean': 0.4262274212484264,
  'std': 0.49453115767286265,
  'se': 0.0018599152988425893,
  'base_variable': 'IsUSBased',
  'ground_truth_distribution_type': 'beta',
  'variable': 'The probability that a venture-backed company is based in the US',
  'conditions': [],
  'nat_langs': []},
 'IsTechCompany': {'mean': 0.47340056862384544,
  'std': 0.4992955001903887,
  'se': 0.0018778338331953525,
  'base_variable': 'IsTechCompany',
  'ground_truth_distribution_type': 'beta',
  'variable': 'The probability that a venture-backed company is a technology company',
  'conditions': [],
  'nat_langs': []},
 'TotalRaised': {'mean': 27.092343523063214,
  'std': 187.8983559868531,
  'se': 0.7066794912819231,
  'mean_lognormal': 26.633789161101923,
  'std_lognormal': 193.9773166722882,
  'base_variable': 'TotalRaised',
  'ground_truth_distribution_type': 'normal',
  'variable': 'The average total raised in millions USD for venture-backed companies',
  'conditions': [],
  'nat_langs': []

In [10]:
results['abs_error_from_mean'] = np.abs(results['ground_truth'] - results['mean'])
results['abs_error_from_median'] = np.abs(results['ground_truth'] - results['median'])
results['abs_error_from_mode'] = np.abs(results['ground_truth'] - results['mode'])

# Determine quartile of ground truth

In [11]:
def get_quartiles_from_gaussian(mean, std):
    return normal_dist.ppf([0.25, 0.5, 0.75], mean, std)


def get_quartiles_from_beta(alpha, beta_param):
    res = beta_dist.ppf([0.25, 0.5, 0.75], alpha, beta_param)
    return res


def get_quartiles_from_lognormal(mu, sigma):
    return np.exp(stats.norm.ppf([0.25, 0.5, 0.75], mu, sigma))


def determine_quartile_of_gt(results):
    # Initialize the 'quartile_of_gt' column with default values
    results['quartile_of_gt'] = np.nan 

    for index, row in results.iterrows():
        if row['fitted_distribution_type'] == 'normal' or row['fitted_distribution_type'] == 'gaussian':
            if row['mean'] is not None:
                quartiles = get_quartiles_from_gaussian(row['mu'], row['sigma'])
        elif row['fitted_distribution_type'] == 'beta' or row['fitted_distribution_type'] == 'binomial':
            # print("Alpha: ", row['alpha'], "Beta: ", row['beta'])
            try: 
                alpha = row['alpha']
                beta = row['beta']
            except:
                alpha = row['a']
                beta = row['b']

            if alpha == 0: 
                alpha = 0.00001
            if beta == 0:
                beta = 0.00001
            if alpha == 0:
                alpha = 0.00001
            quartiles = get_quartiles_from_beta(alpha, beta)
        elif row['fitted_distribution_type'] == 'lognormal':
            if 'sigma' in row:
                quartiles = get_quartiles_from_lognormal(row['mu'], row['sigma'])
            else: 
                print(row)
                quartiles = get_quartiles_from_lognormal(row['mu'], row['std']) 
        else:
            if row['ground_truth_distribution_type'] == 'gaussian':
                quartiles = get_quartiles_from_gaussian(row['mu'], row['sigma'])
            elif row['ground_truth_distribution_type'] == 'beta':
                try: 
                    alpha = row['alpha']
                    beta = row['beta']
                except:
                    alpha = row['a']
                    beta = row['b']

                if alpha == 0: 
                    alpha = 0.00001
                if beta == 0:
                    beta = 0.00001
                quartiles = get_quartiles_from_beta(alpha, beta)
            else:
                raise ValueError("Unknown distribution type: ", row['ground_truth_distribution_type'])
            
        if row['ground_truth'] < quartiles[0]: 
            quartile_of_gt = 1
        elif row['ground_truth'] > quartiles[0] and row['ground_truth'] < quartiles[1]:
            quartile_of_gt = 2
        elif row['ground_truth'] > quartiles[1] and row['ground_truth'] < quartiles[2]:
            quartile_of_gt = 3
        elif row['ground_truth'] > quartiles[2]:
            quartile_of_gt = 4
        else: 
            quartile_of_gt = None
        results.at[index, 'quartile_of_gt'] = quartile_of_gt
    return results


results = determine_quartile_of_gt(results)

/var/folders/yt/njjd4lrj39q3x45zw301h6h80000gn/T/ipykernel_23257/1338078943.py:11: RuntimeWarning: overflow encountered in exp
  return np.exp(stats.norm.ppf([0.25, 0.5, 0.75], mu, sigma))


# Compute stdevs 

In [12]:
# Compute std for each distribution type
def compute_normal_std(mu, sigma):
    return sigma

def compute_beta_std(a, b):
    return stats.beta.std(a, b)

def compute_lognormal_std(mu, sigma):
    """
    Compute std of a Lognormal distribution.
    
    For Lognormal(mu, sigma):
    Var(X) = (exp(sigma^2) - 1) * exp(2*mu + sigma^2)
    Std(X) = sqrt(Var(X))
    """
    variance = (np.exp(sigma**2) - 1) * np.exp(2*mu + sigma**2)
    return np.sqrt(variance)

# Compute std for all rows
results['std'] = np.nan

results.loc[beta_vars, 'std'] = results.loc[beta_vars].apply(
    lambda row: compute_beta_std(row['a'], row['b']), axis=1)

results.loc[lognorm_vars, 'std'] = results.loc[lognorm_vars].apply(
    lambda row: compute_lognormal_std(row['mu'], row['sigma']), axis=1)

results.loc[norm_vars, 'std'] = results.loc[norm_vars].apply(
    lambda row: compute_normal_std(row['mu'], row['sigma']), axis=1)

/var/folders/yt/njjd4lrj39q3x45zw301h6h80000gn/T/ipykernel_23257/2712803009.py:16: RuntimeWarning: overflow encountered in exp
  variance = (np.exp(sigma**2) - 1) * np.exp(2*mu + sigma**2)
/var/folders/yt/njjd4lrj39q3x45zw301h6h80000gn/T/ipykernel_23257/2712803009.py:16: RuntimeWarning: overflow encountered in scalar multiply
  variance = (np.exp(sigma**2) - 1) * np.exp(2*mu + sigma**2)


# Compute error ratios

In [ ]:
fallback_count = 0
exact_match_count = 0

results['error_ratio_mean'] = np.nan
results['error_ratio_median'] = np.nan
results['error_ratio_mode'] = np.nan
results['std_ratio'] = np.nan
results['associated_baseline_error_mean'] = np.nan
results['associated_baseline_error_median'] = np.nan
results['associated_baseline_error_mode'] = np.nan
results['associated_baseline_std'] = np.nan

for idx, row in results.iterrows():
    if "statistical" not in row["approach"]:
        # First try: Match baselines with same variable, sample_size=5, and distribution type
        baselines = results[
            (results["approach"].str.contains("statistical", na=False)) & 
            (results["sample_size"] == "5") &
            (results["variable"] == row["variable"]) & 
            (results["ground_truth_distribution_type"] == row["fitted_distribution_type"])
        ]
        
        # Fallback: If no exact match, use any available baseline for this variable
        if len(baselines) == 0:
            baselines = results[
                (results["approach"].str.contains("statistical", na=False)) & 
                (results["sample_size"] == "5") &
                (results["variable"] == row["variable"])
            ]
            if len(baselines) > 0:
                fallback_count += 1
        else:
            exact_match_count += 1
        
        if len(baselines) == 0:
            continue
            
        baseline_avg_error_mean = baselines["abs_error_from_mean"].mean()
        baseline_avg_error_median = baselines["abs_error_from_median"].mean()
        baseline_avg_error_mode = baselines["abs_error_from_mode"].mean()
        baseline_avg_std = baselines["std"].mean()

        error_ratio_mean = row["abs_error_from_mean"] / baseline_avg_error_mean 
        error_ratio_median = row["abs_error_from_median"] / baseline_avg_error_median
        error_ratio_mode = row["abs_error_from_mode"] / baseline_avg_error_mode
        std_ratio = row["std"] / baseline_avg_std

        results.at[idx, 'associated_baseline_error_mean'] = baseline_avg_error_mean
        results.at[idx, 'associated_baseline_error_median'] = baseline_avg_error_median
        results.at[idx, 'associated_baseline_error_mode'] = baseline_avg_error_mode
        results.at[idx, 'associated_baseline_std'] = baseline_avg_std
        results.at[idx, 'error_ratio_mean'] = error_ratio_mean
        results.at[idx, 'error_ratio_median'] = error_ratio_median
        results.at[idx, 'error_ratio_mode'] = error_ratio_mode
        results.at[idx, 'std_ratio'] = std_ratio
    
print(f"\nExact distribution type matches: {exact_match_count}")
print(f"Fallback to any available baseline: {fallback_count}")


Exact distribution type matches: 301
Fallback to any available baseline: 4


In [19]:
results

,variable_name,variable,ground_truth,ground_truth_distribution_type,fitted_distribution_type,a,b,mu,sigma,temperature,...,median,mode,abs_error_from_mean,abs_error_from_median,abs_error_from_mode,std,error_ratio_mean,error_ratio_median,error_ratio_mode,std_ratio
0,The average number of employees for venture-ba...,Employees,52.925986,normal,lognormal,NaN,NaN,3.910,0.5,0.5,...,49.898952,38.861343,3.616934,3.027034,14.064643,30.134004,0.0,0.0,0.265742,0.0
1,The average number of employees for venture-ba...,Employees,52.925986,normal,lognormal,NaN,NaN,3.900,0.5,0.5,...,49.402449,38.474666,3.054323,3.523537,14.451320,29.834165,0.0,0.0,0.273048,0.0
2,The average number of employees for venture-ba...,Employees,52.925986,normal,lognormal,NaN,NaN,3.900,0.5,0.5,...,49.402449,38.474666,3.054323,3.523537,14.451320,29.834165,0.0,0.0,0.273048,0.0
3,The average number of employees for venture-ba...,Employees,52.925986,normal,lognormal,NaN,NaN,3.912,0.6,medium,...,49.998850,34.883014,6.933505,2.927136,18.042972,39.404163,0.0,0.0,0.340910,0.0
4,The average number of employees for venture-ba...,Employees,52.925986,normal,lognormal,NaN,NaN,4.605,0.8,medium,...,99.982983,52.720269,84.763356,47.056997,0.205716,130.367951,0.0,0.0,0.003887,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10501,The average number of employees for venture-ba...,hard_19,44.274947,beta,beta,20.0,12.0,NaN,NaN,NaN,...,0.627635,0.633333,43.649947,43.647313,43.641614,0.084275,NaN,NaN,NaN,NaN
10502,The average number of employees for venture-ba...,hard_19,44.274947,beta,beta,16.0,16.0,NaN,NaN,NaN,...,0.500000,0.500000,43.774947,43.774947,43.774947,0.087039,NaN,NaN,NaN,NaN
10503,The average number of employees for venture-ba...,hard_19,44.274947,beta,beta,17.0,15.0,NaN,NaN,NaN,...,0.531909,0.533333,43.743697,43.743038,43.741614,0.086869,NaN,NaN,NaN,NaN
10504,The average number of employees for venture-ba...,hard_19,44.274947,beta,beta,18.0,14.0,NaN,NaN,NaN,...,0.563818,0.566667,43.712447,43.711129,43.708281,0.086356,NaN,NaN,NaN,NaN


# Sanity check error ratio averages by model family

In [14]:
# Diagnostic: Check what baselines are actually being used for error ratio calculation

# Pick one LLM row to trace through
sample_llm_row = results[~results['approach'].str.contains('statistical', na=False)].iloc[0]

print("LLM Row:")
print(f"  Variable: {sample_llm_row['variable']}")
print(f"  Fitted dist: {sample_llm_row['fitted_distribution_type']}")
print(f"  Ground truth: {sample_llm_row['ground_truth']}")
print(f"  Mean: {sample_llm_row['mean']}")
print(f"  Abs error from mean: {sample_llm_row['abs_error_from_mean']}")

# Find matching baselines (same logic as error ratio calculation)
baselines = results[
    (results["approach"].str.contains("statistical", na=False)) & 
    (results["sample_size"] == "5") &
    (results["variable"] == sample_llm_row["variable"]) & 
    (results["ground_truth_distribution_type"] == sample_llm_row["fitted_distribution_type"])
]

print(f"\nMatching baselines (exact): {len(baselines)}")

# If no exact match, try fallback
if len(baselines) == 0:
    baselines = results[
        (results["approach"].str.contains("statistical", na=False)) & 
        (results["sample_size"] == "5") &
        (results["variable"] == sample_llm_row["variable"])
    ]
    print(f"Matching baselines (fallback): {len(baselines)}")
    
if len(baselines) > 0:
    print("\nBaseline details:")
    for idx, baseline in baselines.iterrows():
        print(f"  Dist type: {baseline['ground_truth_distribution_type']}, mu={baseline.get('mu')}, sigma={baseline.get('sigma')}, mean={baseline.get('mean')}, abs_error={baseline.get('abs_error_from_mean')}")
    
    avg_error = baselines["abs_error_from_mean"].mean()
    print(f"\nAverage baseline error: {avg_error}")
    print(f"LLM error: {sample_llm_row['abs_error_from_mean']}")
    print(f"Error ratio: {sample_llm_row['abs_error_from_mean'] / avg_error}")

LLM Row:
  Variable: Employees
  Fitted dist: lognormal
  Ground truth: 52.925985886353764
  Mean: 56.542920238322985
  Abs error from mean: 3.6169343519692205

Matching baselines (exact): 25

Baseline details:
  Dist type: lognormal, mu=1523.3771433989734, sigma=32084.390634496587, mean=inf, abs_error=inf
  Dist type: lognormal, mu=45.14103134517891, sigma=70.21250464072615, mean=inf, abs_error=inf
  Dist type: lognormal, mu=985.1607509002096, sigma=10106.426303342863, mean=inf, abs_error=inf
  Dist type: lognormal, mu=10.074082234421471, sigma=6.405992943440954, mean=19325663240106.98, abs_error=19325663240054.055
  Dist type: lognormal, mu=142.23819886700065, sigma=469.4047490363579, mean=inf, abs_error=inf
  Dist type: lognormal, mu=170.96133187886673, sigma=837.7128338958755, mean=inf, abs_error=inf
  Dist type: lognormal, mu=63.38325123180238, sigma=129.62973986093232, mean=inf, abs_error=inf
  Dist type: lognormal, mu=94.78749326955781, sigma=253.5410568953366, mean=inf, abs_err

In [15]:
# Check the aggregated error ratios to see what's happening
print("\nError ratios for all LLM rows with fitted_distribution_type=='lognormal':")
lognorm_llm = results[
    (results['fitted_distribution_type'] == 'lognormal') & 
    (~results['approach'].str.contains('statistical', na=False))
]
print(f"Number of LLM lognormal predictions: {len(lognorm_llm)}")
print(f"\nError ratio stats:")
print(lognorm_llm['error_ratio_mean'].describe())

print("\n\nNow check if there are any inf or very large values:")
print(f"Max error_ratio_mean: {lognorm_llm['error_ratio_mean'].max()}")
print(f"Number of inf values: {np.isinf(lognorm_llm['error_ratio_mean']).sum()}")
print(f"Number of very large (>100): {(lognorm_llm['error_ratio_mean'] > 100).sum()}")

print("\n\nShow some examples with very large ratios:")
large_ratios = lognorm_llm[lognorm_llm['error_ratio_mean'] > 10].head()
for idx, row in large_ratios.iterrows():
    print(f"\nVariable: {row['variable']}, error_ratio: {row['error_ratio_mean']:.2f}")
    print(f"  LLM error: {row['abs_error_from_mean']}, ground_truth: {row['ground_truth']}, mean: {row['mean']}")


Error ratios for all LLM rows with fitted_distribution_type=='lognormal':
Number of LLM lognormal predictions: 200

Error ratio stats:
count    200.000000
mean       0.022996
std        0.215616
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        2.770322
Name: error_ratio_mean, dtype: float64


Now check if there are any inf or very large values:
Max error_ratio_mean: 2.7703216263022314
Number of inf values: 0
Number of very large (>100): 0


Show some examples with very large ratios:


In [16]:
# Check what's happening with easy_11
print("Variable: easy_11")
print(f"Ground truth (gaussian): {variables['easy_11']['mean']}")
if 'mean_lognormal' in variables['easy_11']:
    print(f"Ground truth (lognormal): {variables['easy_11']['mean_lognormal']}")

# Get baselines for this variable
baselines_easy11 = results[
    (results["approach"].str.contains("statistical", na=False)) & 
    (results["sample_size"] == "5") &
    (results["variable"] == "easy_11") &
    (results["ground_truth_distribution_type"] == "lognormal")
]

print(f"\nNumber of baselines: {len(baselines_easy11)}")
if len(baselines_easy11) > 0:
    print(f"Baseline mean errors: mean={baselines_easy11['abs_error_from_mean'].mean():.6f}")
    print(f"Baseline error range: [{baselines_easy11['abs_error_from_mean'].min():.6f}, {baselines_easy11['abs_error_from_mean'].max():.6f}]")
    
    print("\nSample baseline predictions:")
    for idx, row in baselines_easy11.head(5).iterrows():
        print(f"  mean={row['mean']:.4f}, error={row['abs_error_from_mean']:.4f}")

print("\nLLM predictions for this variable:")
llm_easy11 = results[
    (~results["approach"].str.contains("statistical", na=False)) & 
    (results["variable"] == "easy_11") &
    (results["fitted_distribution_type"] == "lognormal")
]
for idx, row in llm_easy11.head(5).iterrows():
    print(f"  {row['approach']}: mean={row['mean']:.4f}, error={row['abs_error_from_mean']:.4f}, ratio={row['error_ratio_mean']:.2f}")

Variable: easy_11
Ground truth (gaussian): 0.5381176071084951

Number of baselines: 25
Baseline mean errors: mean=inf
Baseline error range: [8654794.915018, inf]

Sample baseline predictions:
  mean=883693000460448512.0000, error=883693000460448512.0000
  mean=7972819827606573910303924748411567768737783903423395877657614145042938065291151719113390968677756075054831877729864414307431571360340307436601067722712401341025095474198873255093643176979294870012801460109872418677727399755093629209272069329000779851374235311273460653116885946610994098795937005568.0000, error=7972819827606573910303924748411567768737783903423395877657614145042938065291151719113390968677756075054831877729864414307431571360340307436601067722712401341025095474198873255093643176979294870012801460109872418677727399755093629209272069329000779851374235311273460653116885946610994098795937005568.0000
  mean=515350403134807612771019252172384390748105286526610556647077256287440605554123880559611205536617018906058249243315527

In [18]:
results.groupby('approach')[['error_ratio_mean', 'error_ratio_median', 'error_ratio_mode', 'std_ratio']].mean()

,error_ratio_mean,error_ratio_median,error_ratio_mode,std_ratio
approach,,,,
gpt-4o_base_direct_temp0.5,0.074761,0.072269,9.964896,0.276114
meta-llama-3-70b_base_direct_temp0.5,0.056148,0.055870,8.920250,0.292948
meta-llama-3-8b_base_direct_temp0.5,0.143623,0.124913,18.743266,1.626872
o3-mini_base_direct_tempmedium,0.063067,0.062151,8.544746,0.289241
o4-mini_base_direct_tempmedium,0.062111,0.057207,12.408620,0.201350
statistical_baseline_n10,NaN,NaN,NaN,NaN
statistical_baseline_n20,NaN,NaN,NaN,NaN
statistical_baseline_n30,NaN,NaN,NaN,NaN
statistical_baseline_n5,NaN,NaN,NaN,NaN
